<a href="https://colab.research.google.com/github/MurphyKlein/CS4782_final_project/blob/main/notebook/supervised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PatchTST — Supervised Replication (Table 4, Weather dataset)

Replicates the **Supervised PatchTST/42** column of Table 4 from:
> *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers* (ICLR 2023)

**Config (from paper):**
- Look-back window `L = 336`, patch length `P = 16`, stride `S = 8` → **42 patches**
- Channel-independence: each of the 21 Weather channels processed independently
- BatchNorm (not LayerNorm), instance normalisation, learnable positional encoding
- Prediction horizons `T ∈ {96, 192, 336, 720}`, metric: MSE + MAE

**Paper targets (Weather):**

| T   | MSE   | MAE   |
|-----|-------|-------|
| 96  | 0.152 | 0.199 |
| 192 | 0.197 | 0.243 |
| 336 | 0.249 | 0.283 |
| 720 | 0.320 | 0.335 |

## Cell 1 — Imports & reproducibility

In [1]:
import os, math, json, time, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 2021
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : Tesla T4


## Cell 2 — Configuration

In [2]:
from google.colab import drive
import os

# Standard simple mount
drive.mount('/content/drive')

# Let's check if the project folder exists
project_path = '/content/drive/MyDrive/DL_Final_Project'
if os.path.exists(project_path):
    print(f"Success! Connected to: {project_path}")
    print("Folders found:", os.listdir(project_path))
else:
    print(f"Connected to Drive, but could not find {project_path}")

# Make this GitHub repository importable in Colab.
# Colab can open the notebook from GitHub without cloning the rest of the repo.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MurphyKlein/CS4782_final_project.git"
REPO_DIR = Path("/content/CS4782_final_project")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Repository imports enabled from: {REPO_DIR}")


Mounted at /content/drive
Success! Connected to: /content/drive/MyDrive/DL_Final_Project
Folders found: ['fine.ipynb', 'lin.ipynb', 'weather_data', 'checkpoints (1)', 'results (1)', 'checkpoints', 'results', 'patchtst_supervised.ipynb', 'Untitled0.ipynb']
Repository imports enabled from: /content/CS4782_final_project


In [3]:
import os

weather_data_path = os.path.join(project_path, 'weather_data')

print(f"Checking for weather_data directory at: {weather_data_path}")
if os.path.exists(weather_data_path):
    print(f"'weather_data' directory found. Contents: {os.listdir(weather_data_path)}")
    print("If this list is empty or doesn't contain CSV files, please ensure your weather data files are uploaded here.")
else:
    print(f"'weather_data' directory NOT found at {weather_data_path}.")
    print("Please create a folder named 'weather_data' inside your 'DL_Final_Project' directory in Google Drive and upload your CSV files there.")
    print("Alternatively, if your data is directly in 'DL_Final_Project', you might need to adjust `WEATHER_DIR` in Cell 2.")


Checking for weather_data directory at: /content/drive/MyDrive/DL_Final_Project/weather_data
'weather_data' directory found. Contents: ['mpi_roof_2020a.csv', 'mpi_roof_2020b.csv', 'mpi_roof_2021a.csv', 'mpi_roof_2021b.csv', 'mpi_roof_2022a.csv', 'mpi_roof_2022b.csv', 'mpi_roof_2023a.csv', 'mpi_roof_2023b.csv', 'mpi_roof_2024.csv', 'mpi_roof_2025.csv', 'mpi_roof.csv']
If this list is empty or doesn't contain CSV files, please ensure your weather data files are uploaded here.


In [4]:
import math
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/DL_Final_Project')
WEATHER_DIR = BASE_DIR / 'weather_data'
CKPT_DIR    = BASE_DIR / 'checkpoints'     # model checkpoints saved here
RESULTS_DIR = BASE_DIR / 'results'         # JSON results saved here
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── PatchTST/42 hyperparameters (Appendix A.1.4) ─────────────────────
L       = 336    # look-back window
P       = 16     # patch length
S       = 8      # stride

D_MODEL  = 128
N_HEADS  = 16
N_LAYERS = 3
D_FF     = 256
DROPOUT  = 0.2

# ── Training hyperparameters ──────────────────────────────────────────
BATCH_SIZE  = 128
LR          = 1e-4
MAX_EPOCHS  = 100
PATIENCE    = 10
LR_PATIENCE = 5
LR_MIN      = 1e-6

# ── Prediction horizons to sweep ─────────────────────────────────────
HORIZONS = [96, 192, 336, 720]

# ── Data split ratios ─────────────────────────────────────────────────
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1

print(f'Config set to: {WEATHER_DIR}')
N_patches = math.floor((L - P) / S) + 2
print(f'N patches = {N_patches}')

Config set to: /content/drive/MyDrive/DL_Final_Project/weather_data
N patches = 42


## Cell 3 — Load & merge Weather data

In [5]:
from final_supervised.data_proc import load_weather

try:
    weather_data = load_weather(WEATHER_DIR)
    M = weather_data.shape[1]
    print(f"Number of channels (M) = {M}")
except Exception as e:
    print(f"Error loading data: {e}")

Searching for CSVs in: /content/drive/MyDrive/DL_Final_Project/weather_data
Contents of directory: ['mpi_roof_2020a.csv', 'mpi_roof_2020b.csv', 'mpi_roof_2021a.csv', 'mpi_roof_2021b.csv', 'mpi_roof_2022a.csv', 'mpi_roof_2022b.csv', 'mpi_roof_2023a.csv', 'mpi_roof_2023b.csv', 'mpi_roof_2024.csv', 'mpi_roof_2025.csv', 'mpi_roof.csv']
  Loaded mpi_roof.csv                             shape=(17403, 22)
  Loaded mpi_roof_2020a.csv                       shape=(26200, 22)
  Loaded mpi_roof_2020b.csv                       shape=(26496, 22)
  Loaded mpi_roof_2021a.csv                       shape=(26064, 22)
  Loaded mpi_roof_2021b.csv                       shape=(26496, 22)
  Loaded mpi_roof_2022a.csv                       shape=(26070, 22)
  Loaded mpi_roof_2022b.csv                       shape=(26415, 22)
  Loaded mpi_roof_2023a.csv                       shape=(26061, 22)
  Loaded mpi_roof_2023b.csv                       shape=(26639, 22)
  Loaded mpi_roof_2024.csv                        shap

## Cell 4 — Dataset & DataLoader

In [6]:
from final_supervised.data_proc import build_loaders

# Quick sanity-check with T=96
if "weather_data" in locals() or "weather_data" in globals():
    _tr, _va, _te, _sc = build_loaders(
        weather_data,
        L,
        T=96,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        batch_size=BATCH_SIZE,
    )
    x0, y0 = next(iter(_tr))
    print(f"x shape : {tuple(x0.shape)}  (batch, channels, L)")
    print(f"y shape : {tuple(y0.shape)}  (batch, channels, T)")
    print(f"Train batches={len(_tr)}  Val={len(_va)}  Test={len(_te)}")
else:
    print("weather_data is not defined. Please fix the data loading step.")

x shape : (128, 21, 336)  (batch, channels, L)
y shape : (128, 21, 96)  (batch, channels, T)
Train batches=1818  Val=257  Test=517


## Cell 5 — Patching utility

In [7]:
from final_supervised.models import make_patches

# Verify expected N
_dummy = torch.zeros(2, M, L)
_p     = make_patches(_dummy, P, S)
print(f"make_patches test: input (2, {M}, {L}) -> patches {tuple(_p.shape)}")
print(f"Expected N = {math.floor((L-P)/S)+2},  Got N = {_p.shape[1]}")

make_patches test: input (2, 21, 336) -> patches (42, 42, 16)
Expected N = 42,  Got N = 42


## Cell 6 — Model definition

In [8]:
from final_supervised.models import PatchTST

# Quick parameter count
_tmp = PatchTST(M=M, L=L, T=96, P=P, S=S,
                d_model=D_MODEL, n_heads=N_HEADS,
                n_layers=N_LAYERS, d_ff=D_FF)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"PatchTST/42  N={_tmp.N} patches  params={n_params:,}")
del _tmp

PatchTST/42  N=42 patches  params=921,184


## Cell 7 — Training & evaluation helpers

In [9]:
from final_supervised.train_val import TrainingConfig, train_one_horizon

print("Training pipeline helpers imported.")

Training pipeline helpers imported.


## Cell 8 — Checkpoint utilities

In [10]:
from final_supervised.utils import load_checkpoint, save_checkpoint, save_results

# The notebook still owns CKPT_DIR, RESULTS_DIR, and DEVICE, and passes them explicitly.
print("Checkpoint utilities imported.")
print(f"Checkpoints -> {CKPT_DIR}")
print(f"Results     -> {RESULTS_DIR}")

Checkpoint utilities imported.
Checkpoints -> /content/drive/MyDrive/DL_Final_Project/checkpoints
Results     -> /content/drive/MyDrive/DL_Final_Project/results


## Cell 9 — Train one horizon (core training loop)

In [11]:
TRAINING_CONFIG = TrainingConfig(
    L=L,
    P=P,
    S=S,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_min=LR_MIN,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)

print("Training config defined.")

Training config defined.


## Cell 10 — Run all horizons

In [12]:
# ── Sweep over all four prediction horizons ───────────────────────────
# Each horizon trains an independent model, saves its best checkpoint,
# and appends its test results to *all_results*.

all_results = {}   # T → {mse, mae, best_epoch, history}

for T in HORIZONS:
    result = train_one_horizon(
        weather_data,
        M,
        T,
        config=TRAINING_CONFIG,
        ckpt_dir=CKPT_DIR,
        device=DEVICE,
    )
    all_results[str(T)] = {
        'mse'        : result['mse'],
        'mae'        : result['mae'],
        'best_epoch' : result['best_epoch'],
        'best_val_mse': result['best_val_mse'],
    }

# Persist results to disk
save_results(all_results, RESULTS_DIR, fname='results_supervised.json')

print('\nAll horizons complete.')


  Supervised PatchTST/42   T = 96
  L=336  P=16  S=8  N=42 patches
  Samples  train=232,638  val=32,864  test=66,162
  Parameters: 921,184


KeyboardInterrupt: 

## Cell 11 — Results summary & comparison with paper

In [ ]:
# ── Paper targets for Supervised PatchTST/42, Weather dataset ─────────
paper = {
    96 : (0.152, 0.199),
    192: (0.197, 0.243),
    336: (0.249, 0.283),
    720: (0.320, 0.335),
}

print('\n' + '='*66)
print('  Supervised PatchTST/42 — Weather  (Table 4 replication)')
print('='*66)
print(f"{'T':>5}  "
      f"{'Paper MSE':>10}  {'Ours MSE':>10}  "
      f"{'Paper MAE':>10}  {'Ours MAE':>10}  "
      f"{'Best ep':>7}")
print('-'*66)

for T in HORIZONS:
    r        = all_results[str(T)]
    p_mse, p_mae = paper[T]
    mse_gap  = r['mse'] - p_mse
    mae_gap  = r['mae'] - p_mae
    mse_sym  = '▲' if mse_gap > 0 else '▼'
    mae_sym  = '▲' if mae_gap > 0 else '▼'
    print(
        f"  {T:5d}  "
        f"  {p_mse:>8.3f}    {r['mse']:>8.3f} {mse_sym}{abs(mse_gap):.3f}  "
        f"  {p_mae:>8.3f}    {r['mae']:>8.3f} {mae_sym}{abs(mae_gap):.3f}  "
        f"  {r['best_epoch']:>5d}"
    )

print('='*66)
print('▲ = above paper target  ▼ = below (better than paper)')

# Also print saved file locations
print(f'\nCheckpoints : {CKPT_DIR}')
print(f'Results JSON: {RESULTS_DIR / "results_supervised.json"}')